In [6]:
!pip install pandas numpy

import pandas as pd
import numpy as np

# Membaca dataset
df = pd.read_csv('paket_a_transaksi_pos_2024_2025.csv')

# Menampilkan informasi awal dataframe
df.info()
df.head()


[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


<class 'pandas.DataFrame'>
RangeIndex: 5030 entries, 0 to 5029
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_transaksi   5030 non-null   str  
 1   tanggal        5030 non-null   str  
 2   kode_gerai     5030 non-null   str  
 3   kode_kasir     4769 non-null   str  
 4   kode_produk    5030 non-null   str  
 5   jumlah         5030 non-null   int64
 6   harga_satuan   5030 non-null   str  
 7   diskon_persen  5030 non-null   int64
 8   status         5030 non-null   str  
dtypes: int64(2), str(7)
memory usage: 353.8 KB


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status
0,TRX0241065,2025-06-02,G09,K011,RMT1402,4,13000,0,SELESAI
1,TRX0240510,2024-12-28,G09,K015,SGR1205,1,21000,0,SELESAI
2,TRX0244483,2024-09-04,G11,K029,RMT1407,3,"7500,00",0,SELESAI
3,TRX0242532,2025-07-30,G05,K001,MKN1103,2,6500,0,SELESAI
4,TRX0243509,26/03/2025,G05,K016,RMT1407,1,6000,0,SELESAI


In [9]:
# 1. Pembersihan Format Harga (mengubah koma jadi titik lalu konversi ke float)
df['harga_satuan'] = df['harga_satuan'].astype(str).str.replace(',', '.').astype(float)

# 2. Penanganan Kuantitas Negatif (mengubah nilai negatif menjadi positif dengan abs())
df.loc[df['jumlah'] < 0, 'jumlah'] = df.loc[df['jumlah'] < 0, 'jumlah'].abs()

# 3. Imputasi Nilai Kosong Kasir (mengisi NaN pada kode_kasir dengan 'K_UNKNOWN')
df.loc[df['kode_kasir'].isna(), 'kode_kasir'] = 'K_UNKNOWN'

# 4. Seleksi Posisi Slicing (5 baris pertama untuk kolom tertentu)
t1_slicing = df.iloc[:5][['id_transaksi', 'tanggal', 'kode_gerai', 'jumlah']]
print("Hasil Slicing Task 1.4:")
display(t1_slicing)

Hasil Slicing Task 1.4:


,id_transaksi,tanggal,kode_gerai,jumlah
0,TRX0241065,2025-06-02,G09,4
1,TRX0240510,2024-12-28,G09,1
2,TRX0244483,2024-09-04,G11,3
3,TRX0242532,2025-07-30,G05,2
4,TRX0243509,26/03/2025,G05,1


In [10]:
# 1. Perhitungan Omzet Kotor
df['Total_Kotor'] = df['harga_satuan'] * df['jumlah']

# 2. Perhitungan Nominal Diskon
df['Nominal_Diskon'] = df['Total_Kotor'] * (df['diskon_persen'] / 100.0)

# 3. Perhitungan Omzet Bersih & Pajak PPN 11%
df['Total_Bersih'] = (df['Total_Kotor'] - df['Nominal_Diskon']) * 1.11

# Menampilkan 5 baris teratas setelah penambahan kolom baru
display(df[['id_transaksi', 'Total_Kotor', 'Nominal_Diskon', 'Total_Bersih']].head())

,id_transaksi,Total_Kotor,Nominal_Diskon,Total_Bersih
0,TRX0241065,52000.0,0.0,57720.0
1,TRX0240510,21000.0,0.0,23310.0
2,TRX0244483,22500.0,0.0,24975.0
3,TRX0242532,13000.0,0.0,14430.0
4,TRX0243509,6000.0,0.0,6660.0


In [11]:
# 1. Filter Multi-Kondisi (Status SELESAI, Gerai G05/G09, Total_Bersih > 100.000)
t3_multi = df[(df['status'] == 'SELESAI') & (df['kode_gerai'].isin(['G05', 'G09'])) & (df['Total_Bersih'] > 100000)]
print(f"Jumlah baris filter multi-kondisi: {len(t3_multi)}")

# 2. Filter dengan Query (Gerai G10 dan jumlah >= 5)
t3_query = df.query("kode_gerai == 'G10' and jumlah >= 5")
display(t3_query.head())

Jumlah baris filter multi-kondisi: 139


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
83,TRX0243880,2024-08-04,G10,K002,MKN1101,12,45500.0,0,SELESAI,546000.0,0.0,606060.0
166,TRX0241592,2024-05-11,G10,K033,SGR1202,6,13000.0,0,SELESAI,78000.0,0.0,86580.0
351,TRX0242686,2024-02-03,G10,K016,BYI1502,12,26000.0,0,SELESAI,312000.0,0.0,346320.0
556,TRX0241396,2025-12-16,G10,K009,RMT1406,12,5500.0,5,SELESAI,66000.0,3300.0,69597.0
879,TRX0243559,2025-09-30,G10,K016,SGR1206,6,47000.0,0,SELESAI,282000.0,0.0,313020.0


In [12]:
# 1. Multi-Column Sorting (kode_gerai A-Z, Total_Bersih tertinggi ke terendah)
df_sorted = df.sort_values(by=['kode_gerai', 'Total_Bersih'], ascending=[True, False])
display(df_sorted.head())

# 2. Top-N Records (5 transaksi dengan Total_Bersih tertinggi)
top5_transaksi = df.nlargest(5, 'Total_Bersih')[['id_transaksi', 'kode_gerai', 'Total_Bersih']]
print("Top 5 Transaksi Tertinggi:")
display(top5_transaksi)

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
244,TRX0243257,2024-11-24,G01,K033,MKN1102,1456,32500.0,20,SELESAI,47320000.0,9464000.0,42020160.0
2956,TRX0241367,2025-05-15,G01,K006,RMT1408,12,50500.0,0,SELESAI,606000.0,0.0,672660.0
3588,TRX0240608,2025-07-25,G01,K006,MKN1101,12,47000.0,0,SELESAI,564000.0,0.0,626040.0
2937,TRX0240631,2024-12-24,G01,K013,RMT1408,12,42000.0,0,SELESAI,504000.0,0.0,559440.0
5,TRX0240658,2025-07-18,G01,K008,RMT1408,12,45500.0,10,SELESAI,546000.0,54600.0,545454.0


Top 5 Transaksi Tertinggi:


,id_transaksi,kode_gerai,Total_Bersih
994,TRX0241002,G11,100363536.0
4985,TRX0244221,G09,66934498.5
467,TRX0243696,G06,53466480.0
3393,TRX0241029,G07,45563391.0
244,TRX0243257,G01,42020160.0


In [13]:
# 1. Laporan Agregasi Per Gerai (Khusus status SELESAI)
df_selesai = df[df['status'] == 'SELESAI']
laporan_gerai = df_selesai.groupby('kode_gerai').agg(
    Jumlah_Transaksi=('id_transaksi', 'count'),
    Total_Unit_Terjual=('jumlah', 'sum'),
    Total_Omzet_Bersih=('Total_Bersih', 'sum'),
    Rata_Rata_Transaksi=('Total_Bersih', 'mean')
).sort_values(by='Total_Omzet_Bersih', ascending=False).round(2)

print("Laporan Agregasi Per Gerai:")
display(laporan_gerai)

# 2. Transformasi Kelompok (Window Function untuk persentase kontribusi gerai)
total_omzet_gerai = df.groupby('kode_gerai')['Total_Bersih'].transform('sum')
df['Kontribusi_Gerai_Persen'] = (df['Total_Bersih'] / total_omzet_gerai) * 100

display(df[['id_transaksi', 'kode_gerai', 'Total_Bersih', 'Kontribusi_Gerai_Persen']].head())

Laporan Agregasi Per Gerai:


,Jumlah_Transaksi,Total_Unit_Terjual,Total_Omzet_Bersih,Rata_Rata_Transaksi
kode_gerai,,,,
G11,398,3170,1.253536e+08,314958.82
G06,383,3103,9.113936e+07,237961.77
G09,379,3027,9.019606e+07,237984.32
G07,384,4284,8.036181e+07,209275.55
G01,450,2635,7.077959e+07,157287.97
G03,372,3274,5.980970e+07,160778.77
G10,401,1742,5.892248e+07,146938.85
G08,408,1700,3.550256e+07,87016.08
G12,406,1010,2.464483e+07,60701.55


,id_transaksi,kode_gerai,Total_Bersih,Kontribusi_Gerai_Persen
0,TRX0241065,G09,57720.0,0.063031
1,TRX0240510,G09,23310.0,0.025455
2,TRX0244483,G11,24975.0,0.018364
3,TRX0242532,G05,14430.0,0.060124
4,TRX0243509,G05,6660.0,0.027750
